# 39 · Total within groups, with each group's share, checked against pandas

## What this checks

**Total within groups** adds up a table by group. This check covers release 0.7.0. The share it checks was
added in release 0.3.0.

Beside each group's total or count, the widget can give its percentage. That is a percentage of the groups that share
the columns named as "within", or of all the groups. This check covers the share only. The sum itself is check 29's.
The count, minimum and maximum have no check yet.

A share that is computed over the wrong groups, or rounded by a different rule, tells a reader the wrong proportion.

## How it is checked

We total each group again with **pandas**, Python's library for tables, which shares no code with Fieldwork. pandas
computes each total over the "within" groups itself, and never reads Fieldwork's totals. The rounding is copied from
Fieldwork's stated rule: one decimal place, with a half rounded up.

The data are check 29's South London table from John Snow's cholera study of 1855. It holds 32 sub-districts'
population and cholera deaths, grouped by water company. Three cases are run on it:

1. Each company's share of the population, and of the deaths.
2. The rows counted by company, each as a share of all rows.
3. The first case again, with one population cell emptied.

## What should happen

1. **Every total and every count** equals pandas' exactly. They are whole numbers.
2. **Every share** equals pandas' exactly, under the stated rounding. Rounding can move a group's total away from 100.
   So each group of shares must total within 0.1 times its number of groups of 100.
3. **An empty cell** makes the group that sums over it unknown. Every share in that set becomes unknown too. The
   other set's shares stay as in case 1.
4. **Each number is named** in the `statistic` column: the value column's name for a total, and "% within ..." or
   "% of all" for a share.

These criteria were fixed before the check was run.

In [ ]:
# Only what this notebook imports. See 00-kernel-check for what this kernel really has.
NEEDED = ["pandas"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested:", NEEDED)
except ImportError:
    print("Not Pyodide; using the environment as installed.")

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_39_group_shares import run

report = run(Path.cwd() / "fixtures" / "group-shares-snow-001.json")
print("agrees:", report["agrees"])
for c in report["comparisons"]:
    print(f"\n{'OK ' if c['agrees'] else 'DIFF'} case {c['case']}: {c['note']}")
    for cell in c["cells"]:
        print(f"  {'OK ' if cell['agrees'] else 'DIFF'} {' · '.join(cell['key'])}: {cell['pandas']} / {cell['fieldwork']}")
    for stratum, s in c["strataTotals"].items():
        print(f"  shares in {stratum} total {s['total']} over {s['groups']} groups")
print("\nempty cell:", report["emptyCell"])

## The result on record

First recorded on 8 November 2026, against release 0.3.0. Rerun on 26 November 2026, against release 0.7.0:
**Agrees**. Every cell is exact.

- **Case 1.** Each company's share of the 486,936 people is 34.4, 61.6 and 3.9. With the rounding, they total 99.9.
  Its share of the 1,514 deaths is 55.7, 43.1 and 1.2.
- **Case 2.** The rows counted by company are 37.5, 50.0 and 12.5 per cent of all 64.
- **Case 3.** With St. Saviour's population emptied, every population share is unknown. The deaths' shares are
  unchanged.

It also agrees in the browser kernel.

Running the cell above repeats the check on the same data.

## Rounding is a rule, not a detail

The share is kept to one decimal place, with a half rounded up. That is why case 1's population shares total 99.9,
not 100.

Python's own rounding works differently. It rounds a half to the even digit, and works on the number as the computer
stores it. A lab that used it would report a difference of rule as a fault. So the cell below records where Python's
rounding would land elsewhere. On this data, it would put no share elsewhere.

In [ ]:
for c in report["comparisons"]:
    print(f"case {c['case']}: round() would differ on {len(c['pythonRoundWouldDiffer'])} share(s)")

## What this does not show

- That a share is the measure a reader needs. A share of rows is not a rate.
- What a share of deaths says about risk. Without the population beside it, it says nothing, which is why case 1
  gives both.
- That one decimal place is enough. A set of shares can total 99.9 or 100.1, which the check allows and does not
  judge.

In [ ]:
for item in report["notEstablished"]:
    print("-", item)
print()
print("fixture :", report["fixture"]["id"], report["fixture"]["sha256"][:16])
print("pandas  :", report["environment"]["pandas"])
print("runtime :", report["environment"]["runtime"])